In [18]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")


  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total


In [19]:
# Google Drive Mount + Output Directory
# =================================================================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ================================================================
# V18 UNIFIED RE-RUN: 29 Pathways + 196 Genes + donor_id
# All results to version18-analysis-v2/
# ================================================================

In [42]:
# ================================================================
# V18 UNIFIED RE-RUN: 29 Pathways + 196 Genes + donor_id
# All results to version18-analysis-v2/
# ================================================================
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, spearmanr
import os, time, warnings
warnings.filterwarnings('ignore')

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
RESULTS_V2 = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2'
os.makedirs(RESULTS_V2, exist_ok=True)

print('Loading h5ad...')
adata = sc.read_h5ad(DATA_PATH)
print(f'Loaded: {adata.shape[0]:,} cells, {adata.shape[1]:,} genes')

# donor_id extraction
adata.obs['donor_id'] = adata.obs['sample'].str.split('_').str[1]
adata.obs['Stage'] = adata.obs['Stage'].astype(str)

col_donor = 'donor_id'
col_tissue = 'tissue'
col_stage = 'Stage'
col_lineage = 'major_lineage'

tissue_vals = adata.obs[col_tissue].unique()
liver_name = [t for t in tissue_vals if 'liver' in str(t).lower()][0]
blood_name = [t for t in tissue_vals if 'blood' in str(t).lower()][0]

GROUPS = ['NL', 'IT', 'IA', 'AR', 'CR']
LINEAGES = ['Myeloid', 'CD4_T', 'CD8_T', 'NK', 'B', 'PlasmaB']
COMPARISONS = [('NL','IT'),('NL','IA'),('NL','AR'),('NL','CR'),('IT','IA'),('IA','AR'),('CR','AR')]

# Exclude gdT
gdt_mask = adata.obs[col_lineage].astype(str).str.lower().str.contains('gdt|gamma')
adata = adata[~gdt_mask].copy()
print(f'After gdT exclusion: {adata.shape[0]:,} cells')

# Verify donor counts
print('\nDonor counts:')
for tl, tn in [(liver_name,'Liver'),(blood_name,'Blood')]:
    for g in GROUPS:
        n = adata.obs[(adata.obs[col_tissue]==tl)&(adata.obs[col_stage]==g)][col_donor].nunique()
        print(f'  {tn} {g}: n={n}')

print('\n✅ C0 Setup complete')


Loading h5ad...
Loaded: 243,000 cells, 24,452 genes
After gdT exclusion: 242,219 cells

Donor counts:
  Liver NL: n=6
  Liver IT: n=6
  Liver IA: n=5
  Liver AR: n=3
  Liver CR: n=3
  Blood NL: n=5
  Blood IT: n=5
  Blood IA: n=4
  Blood AR: n=3
  Blood CR: n=3

✅ C0 Setup complete


In [43]:
# ================================================================
# 29 PATHWAY GENE SETS — SINGLE SOURCE OF TRUTH
# Original 26 (C4) + 3 added (antigen_presentation, type1_ifn, tgfb_signaling)
# ================================================================

PATHWAY_GENE_SETS = {
    # --- Immune Effector ---
    'inflammasome': ['NLRP3','CASP1','IL1B','IL18','PYCARD','GSDMD'],
    'cytotoxicity': ['GZMB','GZMA','GZMK','PRF1','GNLY','NKG7','FASLG'],
    'nk_function': ['NCAM1','KLRD1','KLRC2','KLRK1','NCR1','NCR3'],
    'nk_il15_dual': ['IL2RB','IL2RG','JAK1','JAK3','STAT5A','STAT5B','MTOR','AKT1'],

    # --- Immune Regulation ---
    'checkpoint': ['PDCD1','CTLA4','HAVCR2','LAG3','TIGIT','TOX'],
    'exhaustion': ['TOX','PDCD1','HAVCR2','LAG3','TIGIT','ENTPD1'],
    'treg': ['FOXP3','IL2RA','CTLA4','IKZF2','TNFRSF18'],
    'immune_evasion': ['LGALS9','IDO1','CD274','PDCD1LG2','HAVCR2','VSIR','TNFAIP3','IL1RN'],

    # --- T Cell Biology ---
    'naive_t': ['LEF1','TCF7','CCR7','SELL','IL7R'],
    'memory_t': ['IL7R','CD44','EOMES','TBX21','GZMK','CCR7','SELL','TCF7'],
    'tf_programs': ['TBX21','EOMES','GATA3','RORC','BCL6','PRDM1','RUNX3','IRF4','BATF'],
    'tissue_resident': ['ITGAE','ITGA1','CXCR6','CD69','RUNX3','ZNF683'],
    'stemness': ['TCF7','LEF1','MYB','SELL','CCR7','IL7R','BACH2'],

    # --- Signaling ---
    'il15_mtor': ['IL2RB','IL2RG','JAK1','JAK3','STAT5A','STAT5B','MTOR','AKT1','PIK3CA','RPTOR'],

    # --- Metabolism ---
    'glycolysis': ['HK1','HK2','PFKFB3','PKM','LDHA','ENO1','GAPDH','SLC2A1'],
    'oxphos': ['ATP5F1A','ATP5F1B','NDUFA1','NDUFB1','COX5A','UQCRB','SDHB'],
    'mito_dysfunction': ['MT-ATP6','MT-CO1','MT-CO2','MT-CO3','MT-CYB','MT-ND1','MT-ND4','MT-ND5'],
    'metabolic_recovery': ['PPARGC1A','TFAM','NRF1','SIRT1','PRKAA1','PPARA'],

    # --- Cell Fate ---
    'cell_cycle': ['CDK1','CDK2','CDK4','CDK6','CCNB1','CCND1','CCNE1','CDKN1A','CDKN2A','RB1'],
    'proliferation': ['MKI67','TOP2A','PCNA','CDK1','CCNB1'],
    'apoptosis': ['BCL2','BAX','BAK1','CASP3','CASP8','FAS'],
    'senescence': ['CDKN1A','CDKN2A','TP53','RB1','SERPINE1','GLB1','LMNB1'],

    # --- Tissue/Disease ---
    'cancer_associated': ['AFP','GPC3','TERT','TP53','CTNNB1','MYC','VEGFA','MMP9','SPP1'],
    'fibrosis': ['COL1A1','COL3A1','ACTA2','FN1','TGFB1','TIMP1','MMP2','LOX','PDGFRB','LOXL2'],
    'epigenetics': ['DNMT1','DNMT3A','TET2','KDM6A','EZH2','HDAC1','SETDB1','KMT2A'],
    'angiogenesis': ['VEGFA','VEGFB','FLT1','KDR','ANGPT1','ANGPT2','TEK','HIF1A','NOS3'],

    # --- 3 NEW (from C9) ---
    'antigen_presentation': ['HLA-DRA','HLA-DRB1','HLA-DPB1','HLA-DPA1','HLA-DQB1','CD74','B2M','TAP1','TAP2'],
    'type1_ifn': ['MX1','ISG15','STAT1','STAT2','IRF3','IRF7','IFNAR1','OAS1','IFI44L'],
    'tgfb_signaling': ['TGFB1','TGFBR1','TGFBR2','SMAD2','SMAD3','SMAD4','SMAD7'],
}

# Validate against adata
all_pw_genes = set()
print(f'29 Pathway Gene Sets Validation:')
print(f'{"Pathway":<25} {"Defined":>7} {"Found":>5} {"Missing":>7}')
print('-'*50)
total_defined = 0
total_found = 0
for pw, genes in sorted(PATHWAY_GENE_SETS.items()):
    found = [g for g in genes if g in adata.var_names]
    missing = [g for g in genes if g not in adata.var_names]
    all_pw_genes.update(found)
    total_defined += len(genes)
    total_found += len(found)
    flag = ' ⚠️' + str(missing) if missing else ''
    print(f'{pw:<25} {len(genes):>7} {len(found):>5} {len(missing):>7}{flag}')

print(f'\nTotal unique genes across 29 pathways: {len(all_pw_genes)}')
print(f'Total defined: {total_defined}, Found: {total_found}')

# Save gene set definitions
rows = []
for pw, genes in sorted(PATHWAY_GENE_SETS.items()):
    for g in genes:
        rows.append({'pathway': pw, 'gene': g, 'in_h5ad': g in adata.var_names})
geneset_df = pd.DataFrame(rows)
geneset_df.to_csv(os.path.join(RESULTS_V2, 'C0_29pathway_gene_sets.csv'), index=False)
print(f'\n✅ Saved: C0_29pathway_gene_sets.csv')


29 Pathway Gene Sets Validation:
Pathway                   Defined Found Missing
--------------------------------------------------
angiogenesis                    9     9       0
antigen_presentation            9     9       0
apoptosis                       6     6       0
cancer_associated               9     9       0
cell_cycle                     10    10       0
checkpoint                      6     6       0
cytotoxicity                    7     7       0
epigenetics                     8     8       0
exhaustion                      6     6       0
fibrosis                       10    10       0
glycolysis                      8     8       0
il15_mtor                      10    10       0
immune_evasion                  8     8       0
inflammasome                    6     6       0
memory_t                        8     8       0
metabolic_recovery              6     6       0
mito_dysfunction                8     8       0
naive_t                         5     5       0
nk_f

In [44]:
# ================================================================
# Core statistical functions (donor-level)
# ================================================================

def mwu_test(v1, v2):
    if len(v1) < 2 or len(v2) < 2: return np.nan, np.nan
    try:
        stat, p = mannwhitneyu(v1, v2, alternative='two-sided')
        return stat, p
    except: return np.nan, np.nan

def compute_consistency(v1, v2):
    total = len(v1) * len(v2)
    if total == 0: return '0/0'
    m1, m2 = np.mean(v1), np.mean(v2)
    if m2 >= m1:
        count = sum(1 for a in v1 for b in v2 if b > a)
    else:
        count = sum(1 for a in v1 for b in v2 if b < a)
    return f'{count}/{total}'

def sig_label(p):
    if np.isnan(p): return 'NA'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    if p < 0.10: return '\u2020'
    return 'NS'

def get_donor_means_gene(adata, gene, lineage, tissue_label):
    mask = (adata.obs[col_lineage]==lineage) & (adata.obs[col_tissue]==tissue_label)
    sub = adata[mask]
    if gene not in sub.var_names: return pd.DataFrame()
    expr = sub[:,gene].X.toarray().flatten() if hasattr(sub.X,'toarray') else sub[:,gene].X.flatten()
    df = pd.DataFrame({'donor':sub.obs[col_donor].values,'stage':sub.obs[col_stage].values,'expression':expr})
    return df.groupby(['donor','stage'],observed=True)['expression'].mean().reset_index()

def run_gene_comparison(adata, gene, lineage, tissue_label, tissue_name):
    dm = get_donor_means_gene(adata, gene, lineage, tissue_label)
    if len(dm)==0: return []
    results = []
    for s1,s2 in COMPARISONS:
        v1 = dm[dm['stage']==s1]['expression'].values
        v2 = dm[dm['stage']==s2]['expression'].values
        if len(v1)==0 or len(v2)==0: continue
        m1,m2 = np.mean(v1),np.mean(v2)
        pct = ((m2-m1)/m1*100) if abs(m1)>1e-10 else (99999.0 if m2>m1 else -99999.0 if m2<m1 else 0)
        d = '\u2191' if m2>=m1 else '\u2193'
        _,p = mwu_test(v1,v2)
        results.append({'tissue':tissue_name,'gene':gene,'lineage':lineage,
            'comparison':f'{s1}\u2192{s2}','stage1':s1,'stage2':s2,
            'mean_s1':m1,'mean_s2':m2,'pct_change':round(pct,1),
            'direction':d,'p_value':round(p,6) if not np.isnan(p) else np.nan,
            'sig':sig_label(p),'consistency':compute_consistency(v1,v2),
            'n_s1':len(v1),'n_s2':len(v2)})
    return results

print('\u2705 Core functions defined')


✅ Core functions defined


In [45]:
# ================================================================
# C1: Cell Proportions (verification — already done with donor_id)
# ================================================================
os.makedirs(os.path.join(RESULTS_V2, 'C1'), exist_ok=True)

def compute_proportions(tissue_label, tissue_name):
    mask = adata.obs[col_tissue]==tissue_label
    sub = adata.obs[mask]
    counts = sub.groupby([col_donor,col_stage,col_lineage],observed=True).size().reset_index(name='count')
    totals = sub.groupby([col_donor,col_stage],observed=True).size().reset_index(name='total')
    m = counts.merge(totals, on=[col_donor,col_stage])
    m['proportion'] = m['count']/m['total']*100
    m['tissue'] = tissue_name
    return m

liver_props = compute_proportions(liver_name, 'Liver')
blood_props = compute_proportions(blood_name, 'Blood')

# Print summary
print('C1 Cell Proportions (donor-level mean):')
for tissue_name, props in [('Liver',liver_props),('Blood',blood_props)]:
    means = props.groupby([col_stage,col_lineage],observed=True)['proportion'].mean().reset_index()
    print(f'\n{tissue_name}:')
    for g in GROUPS:
        vals = means[means[col_stage]==g].set_index(col_lineage)['proportion']
        parts = [f'{l}={vals.get(l,0):.1f}%' for l in LINEAGES]
        print(f'  {g}: {", ".join(parts)}')

all_props = pd.concat([liver_props, blood_props])
all_props.to_csv(os.path.join(RESULTS_V2, 'C1/C1_proportions.csv'), index=False)
print('\n\u2705 C1 saved')


C1 Cell Proportions (donor-level mean):

Liver:
  NL: Myeloid=1.7%, CD4_T=17.7%, CD8_T=31.8%, NK=43.9%, B=2.5%, PlasmaB=2.5%
  IT: Myeloid=6.0%, CD4_T=23.1%, CD8_T=34.4%, NK=33.7%, B=2.5%, PlasmaB=0.9%
  IA: Myeloid=10.0%, CD4_T=23.1%, CD8_T=41.1%, NK=19.2%, B=6.1%, PlasmaB=0.7%
  AR: Myeloid=1.8%, CD4_T=22.5%, CD8_T=55.1%, NK=18.7%, B=1.5%, PlasmaB=0.4%
  CR: Myeloid=2.5%, CD4_T=26.1%, CD8_T=40.6%, NK=25.4%, B=4.5%, PlasmaB=0.9%

Blood:
  NL: Myeloid=35.1%, CD4_T=17.6%, CD8_T=12.0%, NK=27.7%, B=6.8%, PlasmaB=0.9%
  IT: Myeloid=18.5%, CD4_T=30.0%, CD8_T=13.1%, NK=25.9%, B=12.0%, PlasmaB=0.7%
  IA: Myeloid=14.3%, CD4_T=33.9%, CD8_T=24.1%, NK=15.7%, B=11.1%, PlasmaB=0.9%
  AR: Myeloid=12.4%, CD4_T=34.9%, CD8_T=21.4%, NK=22.4%, B=8.1%, PlasmaB=0.8%
  CR: Myeloid=17.7%, CD4_T=36.4%, CD8_T=15.3%, NK=7.8%, B=21.9%, PlasmaB=0.8%

✅ C1 saved


In [46]:
# ================================================================
# C4: AUCell 29 Pathways — FULL RE-RUN from h5ad
# Stratified sampling → AUCell → donor-level aggregation → MWU
# ================================================================
import subprocess; subprocess.run(['pip','install','pyscenic','-q'], capture_output=True)
from pyscenic.aucell import aucell
from ctxcore.genesig import GeneSignature

os.makedirs(os.path.join(RESULTS_V2, 'C4'), exist_ok=True)
t0 = time.time()

# Stratified sampling (~50K cells, preserving subcluster×stage proportions)
np.random.seed(42)
n_target = 50000
adata.obs['strat_key'] = adata.obs['gut2021_subcluster_v2'].astype(str) + '_' + adata.obs[col_stage]
sampled_idx = []
for group, sub_df in adata.obs.groupby('strat_key', observed=True):
    n = max(1, int(len(sub_df) * n_target / len(adata)))
    sampled_idx.extend(sub_df.sample(n=min(n,len(sub_df)), random_state=42).index.tolist())
adata_sampled = adata[sampled_idx].copy()
print(f'Sampled: {len(adata_sampled):,} cells')
print(adata_sampled.obs[col_stage].value_counts().sort_index())

# Build expression matrix
X = adata_sampled.X.toarray() if hasattr(adata_sampled.X, 'toarray') else adata_sampled.X
expr_df = pd.DataFrame(X, index=adata_sampled.obs_names, columns=adata_sampled.var_names)

# Build GeneSignatures for 29 pathways
signatures = []
for name, genes in sorted(PATHWAY_GENE_SETS.items()):
    available = [g for g in genes if g in adata_sampled.var_names]
    if len(available) >= 3:
        signatures.append(GeneSignature(name=name, gene2weight=available))

print(f'\nRunning AUCell for {len(signatures)} pathways on {len(expr_df):,} cells...')
auc_mtx = aucell(expr_df, signatures, num_workers=4)
print(f'\u2705 AUCell completed: {auc_mtx.shape}')

# Attach metadata
auc_mtx['donor'] = adata_sampled.obs[col_donor].values
auc_mtx['Stage'] = adata_sampled.obs[col_stage].values
auc_mtx['lineage'] = adata_sampled.obs[col_lineage].values
auc_mtx['tissue'] = adata_sampled.obs[col_tissue].values

# Save per-cell scores
auc_mtx.to_csv(os.path.join(RESULTS_V2, 'C4/C4_aucell_percell.csv.gz'), compression='gzip')
print(f'Saved per-cell AUCell scores ({time.time()-t0:.0f}s)')


Sampled: 49,873 cells
Stage
AR     9331
CR     8879
IA    12848
IT    10093
NL     8722
Name: count, dtype: int64

Running AUCell for 29 pathways on 49,873 cells...
✅ AUCell completed: (49873, 29)
Saved per-cell AUCell scores (197s)


In [47]:
# ================================================================
# C4: Donor-level statistics for 29 pathways
# ================================================================
t0 = time.time()
pw_cols = [c for c in auc_mtx.columns if c not in ['donor','Stage','lineage','tissue']]
print(f'Pathway columns: {len(pw_cols)}')

all_results = []
for tissue_label, tissue_name in [(liver_name,'Liver'),(blood_name,'Blood')]:
    tissue_mask = auc_mtx['tissue'] == tissue_label
    for pw in pw_cols:
        for lin in LINEAGES:
            mask = tissue_mask & (auc_mtx['lineage']==lin)
            sub = auc_mtx[mask]
            if len(sub) == 0: continue
            donor_means = sub.groupby(['donor','Stage'])[pw].mean().reset_index()
            for s1,s2 in COMPARISONS:
                v1 = donor_means[donor_means['Stage']==s1][pw].values
                v2 = donor_means[donor_means['Stage']==s2][pw].values
                if len(v1)<2 or len(v2)<2: continue
                m1,m2 = np.mean(v1),np.mean(v2)
                pct = ((m2-m1)/m1*100) if abs(m1)>1e-10 else (99999.0 if m2>m1 else 0)
                _,p = mwu_test(v1,v2)
                all_results.append({'tissue':tissue_name,'lineage':lin,'pathway':pw,
                    'comparison':f'{s1}\u2192{s2}','stage1':s1,'stage2':s2,
                    'mean_s1':round(m1,6),'mean_s2':round(m2,6),
                    'pct_change':round(pct,1),
                    'direction':'\u2191' if m2>=m1 else '\u2193',
                    'p_value':round(p,6),'sig':sig_label(p),
                    'consistency':compute_consistency(v1,v2),
                    'n_s1':len(v1),'n_s2':len(v2)})

c4_df = pd.DataFrame(all_results)
c4_liver = c4_df[c4_df['tissue']=='Liver']
c4_blood = c4_df[c4_df['tissue']=='Blood']

c4_liver.to_csv(os.path.join(RESULTS_V2, 'C4/C4_pathway_liver.csv'), index=False)
c4_blood.to_csv(os.path.join(RESULTS_V2, 'C4/C4_pathway_blood.csv'), index=False)

print(f'C4 complete: Liver={len(c4_liver)}, Blood={len(c4_blood)} ({time.time()-t0:.0f}s)')
print(f'Blood NL\u2192IT n_s2: {sorted(c4_blood[c4_blood["comparison"]=="NL\u2192IT"]["n_s2"].unique())}')

# Summary
nlit = c4_df[c4_df['comparison']=='NL\u2192IT']
print(f'\nNL\u2192IT significant (p<0.05):')
print(f'  Liver: {(c4_liver[c4_liver["comparison"]=="NL\u2192IT"]["p_value"]<0.05).sum()}')
print(f'  Blood: {(c4_blood[c4_blood["comparison"]=="NL\u2192IT"]["p_value"]<0.05).sum()}')
print('\n\u2705 C4 saved')


Pathway columns: 29
C4 complete: Liver=1218, Blood=1218 (5s)
Blood NL→IT n_s2: [np.int64(5)]

NL→IT significant (p<0.05):
  Liver: 15
  Blood: 28

✅ C4 saved


In [48]:
# ================================================================
# C3: 196 genes — Load gene list and run all comparisons
# C5 148-gene donorfix already done → copy. Run remaining 48.
# ================================================================
os.makedirs(os.path.join(RESULTS_V2, 'C3'), exist_ok=True)
os.makedirs(os.path.join(RESULTS_V2, 'C5'), exist_ok=True)
t0 = time.time()

# Load C3 196-gene list
c3_genelist_path = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C3_gene_expression/C3_gene_list_196genes.csv'
c3_genelist = pd.read_csv(c3_genelist_path)
C3_GENES = sorted(c3_genelist.iloc[:,0].unique())
print(f'C3 gene list: {len(C3_GENES)} genes')

# Load C5 148-gene donorfix results
donorfix_dir = '/content/drive/MyDrive/ITLAS/results/version18-analysis/DonorID_Fix'
c5_liver_fix = pd.read_csv(os.path.join(donorfix_dir, 'C5_genes_liver_donorfix.csv'))
c5_blood_fix = pd.read_csv(os.path.join(donorfix_dir, 'C5_genes_blood_donorfix.csv'))
C5_GENES = sorted(c5_liver_fix['gene'].unique())
print(f'C5 genes (donorfix): {len(C5_GENES)}')

# Copy C5 donorfix to v2
c5_liver_fix.to_csv(os.path.join(RESULTS_V2, 'C5/C5_genes_liver.csv'), index=False)
c5_blood_fix.to_csv(os.path.join(RESULTS_V2, 'C5/C5_genes_blood.csv'), index=False)
print('C5 donorfix copied to v2')

# Identify C3-only genes (not in C5)
c3_only = sorted(set(C3_GENES) - set(C5_GENES))
c3_only_found = [g for g in c3_only if g in adata.var_names]
print(f'C3-only genes: {len(c3_only)} total, {len(c3_only_found)} found in h5ad')

# Run C3-only genes
c3only_liver = []
c3only_blood = []
total = len(c3_only_found) * len(LINEAGES)
done = 0
for gene in c3_only_found:
    for lin in LINEAGES:
        c3only_liver.extend(run_gene_comparison(adata, gene, lin, liver_name, 'Liver'))
        c3only_blood.extend(run_gene_comparison(adata, gene, lin, blood_name, 'Blood'))
        done += 1
        if done % 50 == 0:
            print(f'  C3-only: {done}/{total} ({done/total*100:.0f}%) {time.time()-t0:.0f}s')

df_c3only_liver = pd.DataFrame(c3only_liver)
df_c3only_blood = pd.DataFrame(c3only_blood)

# Combine: C5 (148) + C3-only = C3 (196)
c3_liver_all = pd.concat([c5_liver_fix, df_c3only_liver], ignore_index=True)
c3_blood_all = pd.concat([c5_blood_fix, df_c3only_blood], ignore_index=True)

c3_liver_all.to_csv(os.path.join(RESULTS_V2, 'C3/C3_genes_liver.csv'), index=False)
c3_blood_all.to_csv(os.path.join(RESULTS_V2, 'C3/C3_genes_blood.csv'), index=False)

print(f'\nC3 complete: Liver={len(c3_liver_all)}, Blood={len(c3_blood_all)} ({time.time()-t0:.0f}s)')
print(f'C3 unique genes: Liver={c3_liver_all["gene"].nunique()}, Blood={c3_blood_all["gene"].nunique()}')
print('\n\u2705 C3+C5 saved')


C3 gene list: 196 genes
C5 genes (donorfix): 148
C5 donorfix copied to v2
C3-only genes: 115 total, 115 found in h5ad
  C3-only: 50/690 (7%) 70s
  C3-only: 100/690 (14%) 141s
  C3-only: 150/690 (22%) 208s
  C3-only: 200/690 (29%) 278s
  C3-only: 250/690 (36%) 349s
  C3-only: 300/690 (43%) 416s
  C3-only: 350/690 (51%) 486s
  C3-only: 400/690 (58%) 557s
  C3-only: 450/690 (65%) 625s
  C3-only: 500/690 (72%) 695s
  C3-only: 550/690 (80%) 765s
  C3-only: 600/690 (87%) 833s
  C3-only: 650/690 (94%) 903s

C3 complete: Liver=11046, Blood=11046 (958s)
C3 unique genes: Liver=263, Blood=263

✅ C3+C5 saved


In [49]:
# ================================================================
# C7: Gene-gene Spearman correlations (donor-level, n=23)
# Top 15 hub genes from C5
# ================================================================
os.makedirs(os.path.join(RESULTS_V2, 'C7'), exist_ok=True)
t0 = time.time()

# Get top genes by significance frequency
c5_all = pd.concat([c5_liver_fix, c5_blood_fix])
sig_counts = c5_all[c5_all['p_value']<0.05].groupby('gene').size().sort_values(ascending=False)
TOP_GENES = sig_counts.head(15).index.tolist()
print(f'Top 15 genes by significance: {TOP_GENES}')

# Compute donor-level means for top genes across all lineages/tissues
corr_results = []
pairs_tested = 0
for tissue_label, tissue_name in [(liver_name,'Liver'),(blood_name,'Blood')]:
    for lin in LINEAGES:
        # Build donor × gene matrix
        donor_gene_matrix = {}
        for gene in TOP_GENES:
            dm = get_donor_means_gene(adata, gene, lin, tissue_label)
            if len(dm) > 0:
                donor_gene_matrix[gene] = dm.set_index('donor')['expression']

        if len(donor_gene_matrix) < 2: continue
        dgm = pd.DataFrame(donor_gene_matrix).dropna()
        if len(dgm) < 5: continue

        genes_available = list(dgm.columns)
        for i in range(len(genes_available)):
            for j in range(i+1, len(genes_available)):
                g1, g2 = genes_available[i], genes_available[j]
                rho, p = spearmanr(dgm[g1], dgm[g2])
                pairs_tested += 1
                if not np.isnan(rho):
                    corr_results.append({'tissue':tissue_name,'lineage':lin,
                        'gene1':g1,'gene2':g2,'rho':round(rho,3),
                        'p_value':round(p,6),'sig':sig_label(p),'n':len(dgm)})

c7_df = pd.DataFrame(corr_results)
c7_df.to_csv(os.path.join(RESULTS_V2, 'C7/C7_correlations.csv'), index=False)

sig_corr = c7_df[c7_df['p_value']<0.05]
print(f'\nC7 complete: {pairs_tested} pairs tested, {len(sig_corr)} significant ({time.time()-t0:.0f}s)')
print(f'Top 5 correlations:')
print(sig_corr.sort_values('rho', ascending=False).head(5)[['tissue','lineage','gene1','gene2','rho','p_value']].to_string())
print('\n\u2705 C7 saved')


Top 15 genes by significance: ['JAK1', 'TGFBR2', 'MT-ND1', 'TFAM', 'MT-CYB', 'RPTOR', 'MT-ND2', 'DNMT3A', 'MTOR', 'SDHB', 'CD44', 'CD27', 'DNMT1', 'ATM', 'RB1']

C7 complete: 1260 pairs tested, 598 significant (123s)
Top 5 correlations:
     tissue  lineage   gene1   gene2    rho  p_value
677   Blood  Myeloid    TFAM   DNMT1  0.925      0.0
868   Blood    CD8_T  MT-ND1  MT-CYB  0.902      0.0
1050  Blood        B    JAK1  TGFBR2  0.896      0.0
763   Blood    CD4_T  MT-ND1  MT-CYB  0.890      0.0
735   Blood    CD4_T    JAK1  TGFBR2  0.880      0.0

✅ C7 saved


In [50]:
# ================================================================
# C8: Pattern classification from C5 data
# ================================================================
os.makedirs(os.path.join(RESULTS_V2, 'C8'), exist_ok=True)

def classify_patterns(liver_df, blood_df):
    patterns = {}
    # IT-specific: NL→IT sig, NL→IA NS
    for tissue_name, df in [('Liver',liver_df),('Blood',blood_df)]:
        nlit = df[(df['comparison']=='NL\u2192IT')&(df['p_value']<0.05)]
        nlia = df[df['comparison']=='NL\u2192IA']
        nlia_dict = {(r['gene'],r['lineage']):r['p_value'] for _,r in nlia.iterrows()}
        for _,row in nlit.iterrows():
            ia_p = nlia_dict.get((row['gene'],row['lineage']))
            if ia_p is not None and ia_p >= 0.05:
                key = (tissue_name,row['gene'],row['lineage'])
                patterns[key] = {'pattern':'IT_specific','tissue':tissue_name,
                    'gene':row['gene'],'lineage':row['lineage'],
                    'IT_pct':row['pct_change'],'IT_p':row['p_value'],
                    'IA_p':ia_p,'direction':row['direction'],
                    'consistency':row['consistency']}

    # Chronic-persistent: NL→IT sig AND NL→IA sig
    for tissue_name, df in [('Liver',liver_df),('Blood',blood_df)]:
        nlit = df[(df['comparison']=='NL\u2192IT')&(df['p_value']<0.05)]
        nlia = df[(df['comparison']=='NL\u2192IA')&(df['p_value']<0.05)]
        nlia_keys = set(zip(nlia['gene'],nlia['lineage']))
        for _,row in nlit.iterrows():
            key = (tissue_name,row['gene'],row['lineage'])
            if key not in patterns and (row['gene'],row['lineage']) in nlia_keys:
                patterns[key] = {'pattern':'chronic_persistent','tissue':tissue_name,
                    'gene':row['gene'],'lineage':row['lineage'],
                    'IT_pct':row['pct_change'],'IT_p':row['p_value'],
                    'IA_p':'<0.05','direction':row['direction'],
                    'consistency':row['consistency']}

    return pd.DataFrame(patterns.values())

c8_df = classify_patterns(c5_liver_fix, c5_blood_fix)

# Exclude gdT
c8_df = c8_df[~c8_df['lineage'].str.contains('gdT|gdt',case=False)]

it_spec = c8_df[c8_df['pattern']=='IT_specific']
chronic = c8_df[c8_df['pattern']=='chronic_persistent']

print(f'C8 Pattern Classification:')
print(f'  IT-specific: {len(it_spec)} (Liver={len(it_spec[it_spec["tissue"]=="Liver"])}, Blood={len(it_spec[it_spec["tissue"]=="Blood"])})')
print(f'  Chronic-persistent: {len(chronic)} (Liver={len(chronic[chronic["tissue"]=="Liver"])}, Blood={len(chronic[chronic["tissue"]=="Blood"])})')

c8_df.to_csv(os.path.join(RESULTS_V2, 'C8/C8_patterns.csv'), index=False)
it_spec.to_csv(os.path.join(RESULTS_V2, 'C8/C8_IT_specific.csv'), index=False)

# Key gene verification
print('\nKey gene check:')
for g,l,t in [('TOX','CD4_T','Liver'),('LAYN','CD4_T','Liver'),('BCL6','CD8_T','Liver'),('TOX','CD8_T','Liver')]:
    found = it_spec[(it_spec['gene']==g)&(it_spec['lineage']==l)&(it_spec['tissue']==t)]
    print(f'  {"\u2705" if len(found)>0 else "\u274c"} {g}/{l}/{t}')
print('\n\u2705 C8 saved')


C8 Pattern Classification:
  IT-specific: 114 (Liver=52, Blood=62)
  Chronic-persistent: 84 (Liver=32, Blood=52)

Key gene check:
  ✅ TOX/CD4_T/Liver
  ✅ LAYN/CD4_T/Liver
  ✅ BCL6/CD8_T/Liver
  ✅ TOX/CD8_T/Liver

✅ C8 saved


In [51]:
# ================================================================
# FINAL SUMMARY: V18 Unified Re-run
# ================================================================
print('='*70)
print('V18 UNIFIED RE-RUN COMPLETE')
print('='*70)

print(f'\nResults directory: {RESULTS_V2}')
for root, dirs, files in os.walk(RESULTS_V2):
    level = root.replace(RESULTS_V2, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        size = os.path.getsize(os.path.join(root, f))
        print(f'{indent}  {f} ({size:,} bytes)')

print(f'\n--- KEY NUMBERS ---')
print(f'Pathways: {len(pw_cols)} (target: 29)')
print(f'C3 genes: {c3_liver_all["gene"].nunique()} (target: 196)')
print(f'C5 genes: {c5_liver_fix["gene"].nunique()} (target: 148)')
print(f'IT-specific: Liver={len(it_spec[it_spec["tissue"]=="Liver"])}, Blood={len(it_spec[it_spec["tissue"]=="Blood"])}')
print(f'C7 correlations: {len(c7_df)} tested, {len(sig_corr)} significant')

print(f'\n--- DONOR COUNTS (should be 5/5 for Blood IT) ---')
c4_b_nlit = c4_blood[c4_blood['comparison']=='NL\u2192IT']
print(f'C4 Blood NL\u2192IT n_s2: {sorted(c4_b_nlit["n_s2"].unique())}')
c5_b = c5_blood_fix[c5_blood_fix['comparison']=='NL\u2192IT']
print(f'C5 Blood NL\u2192IT n_s2: {sorted(c5_b["n_s2"].unique())}')

print(f'\n--- REMAINING (manual) ---')
print('  C9B: FDR re-calculation (from new C3/C5 p-values)')
print('  C10: Oaxaca-Blinder (independent of donor_id issue)')
print('  Figures 1-7: Re-generate from v2 data')
print('  Manuscript: Update all numbers')
print('\n\u2705 ALL DONE')


V18 UNIFIED RE-RUN COMPLETE

Results directory: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2
version18-analysis-v2/
  C0_29pathway_gene_sets.csv (5,004 bytes)
  C1/
    C1_proportions.csv (12,838 bytes)
  C4/
    C4_aucell_percell.csv.gz (6,143,016 bytes)
    C4_pathway_blood.csv (103,173 bytes)
    C4_pathway_liver.csv (103,531 bytes)
  C3/
    C3_genes_blood.csv (1,083,451 bytes)
    C3_genes_liver.csv (1,082,339 bytes)
  C5/
    C5_genes_blood.csv (563,964 bytes)
    C5_genes_liver.csv (566,713 bytes)
  C7/
    C7_correlations.csv (54,825 bytes)
  C8/
    C8_IT_specific.csv (8,452 bytes)
    C8_patterns.csv (14,895 bytes)

--- KEY NUMBERS ---
Pathways: 29 (target: 29)
C3 genes: 263 (target: 196)
C5 genes: 148 (target: 148)
IT-specific: Liver=52, Blood=62
C7 correlations: 1260 tested, 598 significant

--- DONOR COUNTS (should be 5/5 for Blood IT) ---
C4 Blood NL→IT n_s2: [np.int64(5)]
C5 Blood NL→IT n_s2: [np.int64(5)]

--- REMAINING (manual) ---
  C9B: FDR re-calculati

In [ ]:
### C6 and C9B - RE-RUN

In [53]:
# ================================================================
# C6: Tissue Discrepancy Analysis (from v2 C4 + C5 data)
# ================================================================
os.makedirs(os.path.join(RESULTS_V2, 'C6'), exist_ok=True)

c4_liver = pd.read_csv(os.path.join(RESULTS_V2, 'C4/C4_pathway_liver.csv'))
c4_blood = pd.read_csv(os.path.join(RESULTS_V2, 'C4/C4_pathway_blood.csv'))

nlit_l = c4_liver[c4_liver['comparison'] == 'NL→IT'].copy()
nlit_b = c4_blood[c4_blood['comparison'] == 'NL→IT'].copy()

merged_pw = nlit_l.merge(nlit_b, on=['pathway', 'lineage'], suffixes=('_liver', '_blood'))

tissue_opposite_pw = merged_pw[
    ((merged_pw['pct_change_liver'] > 0) & (merged_pw['pct_change_blood'] < 0)) |
    ((merged_pw['pct_change_liver'] < 0) & (merged_pw['pct_change_blood'] > 0))
].copy()

tissue_opposite_sig = tissue_opposite_pw[
    (tissue_opposite_pw['p_value_liver'] < 0.05) | (tissue_opposite_pw['p_value_blood'] < 0.05)
].sort_values('p_value_liver')

print("="*80)
print("C6: TISSUE-OPPOSITE PATHWAYS at NL→IT (v2)")
print("="*80)
print(f"Total direction-discordant: {len(tissue_opposite_pw)}")
print(f"With at least one sig: {len(tissue_opposite_sig)}")
print(f"\n{'Pathway':<25} {'Lineage':<10} {'L_pct':>7} {'L_p':>8} {'L_sig':>5} | {'B_pct':>7} {'B_p':>8} {'B_sig':>5}")
print("-"*85)
for _, r in tissue_opposite_sig.iterrows():
    print(f"{r['pathway']:<25} {r['lineage']:<10} "
          f"{r['pct_change_liver']:>+6.1f}% {r['p_value_liver']:>8.4f} {r['sig_liver']:>5} | "
          f"{r['pct_change_blood']:>+6.1f}% {r['p_value_blood']:>8.4f} {r['sig_blood']:>5}")

tissue_opposite_sig.to_csv(os.path.join(RESULTS_V2, 'C6/C6_tissue_opposite_pathways.csv'), index=False)

# C5 gene-level
c5_liver = pd.read_csv(os.path.join(RESULTS_V2, 'C5/C5_genes_liver.csv'))
c5_blood = pd.read_csv(os.path.join(RESULTS_V2, 'C5/C5_genes_blood.csv'))

nlit_gl = c5_liver[c5_liver['comparison'] == 'NL→IT'].copy()
nlit_gb = c5_blood[c5_blood['comparison'] == 'NL→IT'].copy()

merged_g = nlit_gl.merge(nlit_gb, on=['gene', 'lineage'], suffixes=('_liver', '_blood'))
tissue_opp_genes = merged_g[
    ((merged_g['pct_change_liver'] > 0) & (merged_g['pct_change_blood'] < 0)) |
    ((merged_g['pct_change_liver'] < 0) & (merged_g['pct_change_blood'] > 0))
]
tissue_opp_sig_genes = tissue_opp_genes[
    (tissue_opp_genes['p_value_liver'] < 0.05) | (tissue_opp_genes['p_value_blood'] < 0.05)
].sort_values('p_value_liver')

print(f"\nC6: TISSUE-OPPOSITE GENES at NL→IT (C5 148 genes)")
print(f"  Direction-discordant: {len(tissue_opp_genes)}")
print(f"  With at least one sig: {len(tissue_opp_sig_genes)}")

tissue_opp_sig_genes.to_csv(os.path.join(RESULTS_V2, 'C6/C6_tissue_opposite_genes.csv'), index=False)
print('\n✅ C6 saved')

C6: TISSUE-OPPOSITE PATHWAYS at NL→IT (v2)
Total direction-discordant: 61
With at least one sig: 9

Pathway                   Lineage      L_pct      L_p L_sig |   B_pct      B_p B_sig
-------------------------------------------------------------------------------------
antigen_presentation      CD4_T       +30.6%   0.0152     * |  -15.1%   0.4206    NS
cell_cycle                Myeloid     +29.9%   0.0303     * |  -31.1%   0.0635     †
checkpoint                CD8_T       +42.2%   0.0411     * |  -51.5%   0.0317     *
exhaustion                CD8_T       +40.3%   0.0411     * |  -49.5%   0.0556     †
senescence                Myeloid     +21.2%   0.0547     † |  -29.4%   0.0159     *
checkpoint                NK          +11.6%   0.3290    NS |  -31.7%   0.0159     *
exhaustion                NK          +12.9%   0.3290    NS |  -31.2%   0.0159     *
senescence                NK          +10.7%   0.4286    NS |  -33.6%   0.0159     *
epigenetics               Myeloid      -5.9%   0.

In [55]:
# ================================================================
# C9B: Stratified FDR (from v2 C3 + C5 p-values)
# ================================================================
from statsmodels.stats.multitest import multipletests

os.makedirs(os.path.join(RESULTS_V2, 'C9B'), exist_ok=True)

def stratified_fdr(df, level='B'):
    df = df.copy()
    df['q_value'] = np.nan

    if level == 'B':
        group_cols = ['comparison', 'tissue', 'lineage']
    elif level == 'A':
        group_cols = ['comparison', 'tissue']
    else:
        group_cols = ['comparison']

    for name, group in df.groupby(group_cols, observed=True):
        valid_mask = group['p_value'].notna()
        valid_idx = group.index[valid_mask]
        if len(valid_idx) < 2:
            continue
        pvals = df.loc[valid_idx, 'p_value'].values
        reject, qvals, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
        df.loc[valid_idx, 'q_value'] = qvals

    return df

# C5 FDR
print("="*60)
print("C9B: Stratified FDR — C5 (148 genes)")
print("="*60)

c5_liver = pd.read_csv(os.path.join(RESULTS_V2, 'C5/C5_genes_liver.csv'))
c5_blood = pd.read_csv(os.path.join(RESULTS_V2, 'C5/C5_genes_blood.csv'))
c5_all = pd.concat([c5_liver, c5_blood], ignore_index=True)
c5_fdr = stratified_fdr(c5_all, level='B')

for comp in ['NL→IT', 'NL→IA', 'NL→AR', 'NL→CR']:
    sub = c5_fdr[c5_fdr['comparison'] == comp]
    nominal = (sub['p_value'] < 0.05).sum()
    survived = (sub['q_value'] < 0.05).sum()
    print(f"  {comp}: nominal={nominal}, FDR survived={survived}")

c5_fdr.to_csv(os.path.join(RESULTS_V2, 'C9B/C5_stratified_FDR.csv'), index=False)

# C3 Blood FDR
print(f"\nC9B: Stratified FDR — C3 Blood")
c3_blood = pd.read_csv(os.path.join(RESULTS_V2, 'C3/C3_genes_blood.csv'))
c3_blood_fdr = stratified_fdr(c3_blood, level='B')

nlit_blood = c3_blood_fdr[c3_blood_fdr['comparison'] == 'NL→IT']
nominal = (nlit_blood['p_value'] < 0.05).sum()
survived = (nlit_blood['q_value'] < 0.05).sum()
print(f"  NL→IT Blood: nominal={nominal}, FDR survived={survived}")

if survived > 0:
    survivors = nlit_blood[nlit_blood['q_value'] < 0.05].sort_values('q_value')
    print(f"\n  Top FDR survivors (Blood NL→IT):")
    for _, r in survivors.head(20).iterrows():
        print(f"    {r['gene']:<12} {r['lineage']:<10} pct={r['pct_change']:>+7.1f}% "
              f"p={r['p_value']:.4f} q={r['q_value']:.4f}")

c3_blood_fdr.to_csv(os.path.join(RESULTS_V2, 'C9B/C3_blood_stratified_FDR.csv'), index=False)

# C4 FDR
print(f"\nC9B: Stratified FDR — C4 (29 pathways)")
c4_liver = pd.read_csv(os.path.join(RESULTS_V2, 'C4/C4_pathway_liver.csv'))
c4_blood = pd.read_csv(os.path.join(RESULTS_V2, 'C4/C4_pathway_blood.csv'))
c4_all = pd.concat([c4_liver, c4_blood], ignore_index=True)
c4_fdr = stratified_fdr(c4_all, level='B')

for comp in ['NL→IT']:
    sub = c4_fdr[c4_fdr['comparison'] == comp]
    nominal = (sub['p_value'] < 0.05).sum()
    survived = (sub['q_value'] < 0.05).sum()
    print(f"  {comp}: nominal={nominal}, FDR survived={survived}")

c4_fdr.to_csv(os.path.join(RESULTS_V2, 'C9B/C4_stratified_FDR.csv'), index=False)

# C7 FDR
print(f"\nC9B: FDR — C7 correlations")
c7_df = pd.read_csv(os.path.join(RESULTS_V2, 'C7/C7_correlations.csv'))

# Global FDR
valid_mask = c7_df['p_value'].notna()
valid_idx = c7_df.index[valid_mask]
if len(valid_idx) > 1:
    pvals = c7_df.loc[valid_idx, 'p_value'].values
    reject, qvals, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
    c7_df['q_global'] = np.nan
    c7_df.loc[valid_idx, 'q_global'] = qvals

# Stratified FDR
c7_df['q_stratified'] = np.nan
for (tissue, lin), group in c7_df.groupby(['tissue', 'lineage']):
    vm = group['p_value'].notna()
    vi = group.index[vm]
    if len(vi) < 2: continue
    _, qv, _, _ = multipletests(c7_df.loc[vi, 'p_value'].values, alpha=0.05, method='fdr_bh')
    c7_df.loc[vi, 'q_stratified'] = qv

global_surv = (c7_df['q_global'] < 0.05).sum()
strat_surv = (c7_df['q_stratified'] < 0.05).sum()
print(f"  Global: {global_surv}/{len(c7_df)} survived")
print(f"  Stratified: {strat_surv}/{len(c7_df)} survived")

c7_df.to_csv(os.path.join(RESULTS_V2, 'C9B/C7_FDR.csv'), index=False)

print('\n✅ C9B saved')

C9B: Stratified FDR — C5 (148 genes)
  NL→IT: nominal=198, FDR survived=0
  NL→IA: nominal=201, FDR survived=0
  NL→AR: nominal=185, FDR survived=0
  NL→CR: nominal=257, FDR survived=0

C9B: Stratified FDR — C3 Blood
  NL→IT Blood: nominal=224, FDR survived=0

C9B: Stratified FDR — C4 (29 pathways)
  NL→IT: nominal=43, FDR survived=0

C9B: FDR — C7 correlations
  Global: 438/1260 survived
  Stratified: 433/1260 survived

✅ C9B saved


In [56]:
# ================================================================
# FINAL COMPLETE SUMMARY (v2)
# ================================================================
print('='*70)
print('V18-v2 ALL LAYERS COMPLETE')
print('='*70)

# File listing
for root, dirs, files in os.walk(RESULTS_V2):
    level = root.replace(RESULTS_V2, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        size = os.path.getsize(os.path.join(root, f))
        print(f'{indent}  {f} ({size:,} bytes)')

print(f'\n--- COMPLETE KEY NUMBERS ---')
print(f'Pathways: 29')
print(f'C3 genes: 263 (C3∪C5 universe)')
print(f'C5 genes: 148')
print(f'C3-only: 115')
print(f'C3∩C5 shared: 81')
print(f'Pathway unique genes: 179')
print(f'IT-specific (C5): Liver=52, Blood=62, Total=114')
print(f'C7 correlations: {len(c7_df)} tested, global FDR survived={global_surv}')
print(f'Blood NL→IT n_s2: 5 (donor-level ✅)')

print(f'\n--- REMAINING ---')
print(f'  C10: Oaxaca-Blinder (existing results valid)')
print(f'  Figures 1-7: Re-generate from v2 data')
print(f'  Manuscript: Update all numbers from v2')
print(f'  S2 Sheet 2: Regenerate with new IT-specific (114)')
print('='*70)

V18-v2 ALL LAYERS COMPLETE
version18-analysis-v2/
  C0_29pathway_gene_sets.csv (5,004 bytes)
  C1/
    C1_proportions.csv (12,838 bytes)
  C4/
    C4_aucell_percell.csv.gz (6,143,016 bytes)
    C4_pathway_blood.csv (103,173 bytes)
    C4_pathway_liver.csv (103,531 bytes)
  C3/
    C3_genes_blood.csv (1,083,451 bytes)
    C3_genes_liver.csv (1,082,339 bytes)
  C5/
    C5_genes_blood.csv (563,964 bytes)
    C5_genes_liver.csv (566,713 bytes)
  C7/
    C7_correlations.csv (54,825 bytes)
  C8/
    C8_IT_specific.csv (8,452 bytes)
    C8_patterns.csv (14,895 bytes)
  C6/
    C6_tissue_opposite_genes.csv (6,602 bytes)
    C6_tissue_opposite_pathways.csv (1,777 bytes)
  C9B/
    C3_blood_stratified_FDR.csv (1,217,115 bytes)
    C4_stratified_FDR.csv (238,967 bytes)
    C5_stratified_FDR.csv (1,267,132 bytes)
    C7_FDR.csv (101,177 bytes)

--- COMPLETE KEY NUMBERS ---
Pathways: 29
C3 genes: 263 (C3∪C5 universe)
C5 genes: 148
C3-only: 115
C3∩C5 shared: 81
Pathway unique genes: 179
IT-specific 

In [ ]:
### top 15 genes 중심으로 분석하여, 추가로 manuscript에서 인용하는 핵심유전자 모두 확장 분석

In [57]:
# ========================================
# C7 EXPANDED: Top 15 + 6-Layer genes + C3-only critical genes
# ========================================

# Top 15 from significance ranking (already computed)
TOP15 = ['JAK1', 'TGFBR2', 'MT-ND1', 'TFAM', 'MT-CYB', 'RPTOR', 'MT-ND2',
         'DNMT3A', 'MTOR', 'SDHB', 'CD44', 'CD27', 'DNMT1', 'ATM', 'RB1']

# 6-Layer model genes (from manuscript Table 6)
LAYER_GENES = ['TGFB1', 'LGALS9', 'LILRB1', 'SIGLEC10', 'AIM2', 'MEFV',  # L1
               'DNMT1', 'DNMT3A', 'TET2',  # L2
               'MTOR', 'LDHA', 'TFAM',  # L3
               'JAK1', 'STAT1', 'SOCS1', 'SOCS3',  # L4
               'TOX', 'TOX2', 'LAYN',  # L5
               'PRDM1', 'RORC']  # L6

# C3-only critical genes (C9B verified)
C3ONLY_CRITICAL = ['MX1', 'ISG15', 'STAT2', 'IRF3', 'IRF7', 'SOCS1', 'SOCS3',
                    'AICDA', 'JCHAIN', 'HLA-DRA', 'HLA-DRB1', 'HLA-DPB1',
                    'HLA-DPA1', 'CD74', 'B2M', 'TAP1', 'IL1RN', 'IL1B']

# Additional key genes from Results
ADDITIONAL = ['TGFBR1', 'TGFBR2', 'IL2RA', 'TYROBP', 'FCER1G', 'GZMB',
              'BCL6', 'CTLA4', 'TIGIT', 'CASP1', 'PYCARD', 'BAK1']

# Combine and deduplicate
C7_GENES = sorted(set(TOP15 + LAYER_GENES + C3ONLY_CRITICAL + ADDITIONAL))
C7_GENES_FOUND = [g for g in C7_GENES if g in adata.var_names]
print(f"C7 expanded gene list: {len(C7_GENES)} defined, {len(C7_GENES_FOUND)} found")
print(f"  Top15: {len(TOP15)}")
print(f"  + Layer genes: {len(set(LAYER_GENES) - set(TOP15))} new")
print(f"  + C3-only: {len(set(C3ONLY_CRITICAL) - set(TOP15) - set(LAYER_GENES))} new")
print(f"  + Additional: {len(set(ADDITIONAL) - set(TOP15) - set(LAYER_GENES) - set(C3ONLY_CRITICAL))} new")

C7 expanded gene list: 58 defined, 58 found
  Top15: 15
  + Layer genes: 16 new
  + C3-only: 16 new
  + Additional: 11 new


In [58]:
# ========================================
# C7 EXPANDED: Re-run correlations with full gene set
# ========================================
t0 = time.time()

corr_results = []
pairs_tested = 0

for tissue_label, tissue_name in [(liver_name, 'Liver'), (blood_name, 'Blood')]:
    for lin in LINEAGES:
        donor_gene_matrix = {}
        for gene in C7_GENES_FOUND:
            dm = get_donor_means_gene(adata, gene, lin, tissue_label)
            if len(dm) > 0:
                donor_gene_matrix[gene] = dm.set_index('donor')['expression']

        if len(donor_gene_matrix) < 2: continue
        dgm = pd.DataFrame(donor_gene_matrix).dropna()
        if len(dgm) < 5: continue

        genes_avail = list(dgm.columns)
        for i in range(len(genes_avail)):
            for j in range(i+1, len(genes_avail)):
                g1, g2 = genes_avail[i], genes_avail[j]
                rho, p = spearmanr(dgm[g1], dgm[g2])
                pairs_tested += 1
                if not np.isnan(rho):
                    corr_results.append({
                        'tissue': tissue_name, 'lineage': lin,
                        'gene1': g1, 'gene2': g2,
                        'rho': round(rho, 3), 'p_value': round(p, 6),
                        'sig': sig_label(p), 'n': len(dgm)
                    })

c7_expanded = pd.DataFrame(corr_results)

# FDR
from statsmodels.stats.multitest import multipletests
valid_idx = c7_expanded.index[c7_expanded['p_value'].notna()]
if len(valid_idx) > 1:
    _, qvals, _, _ = multipletests(c7_expanded.loc[valid_idx, 'p_value'].values, alpha=0.05, method='fdr_bh')
    c7_expanded['q_global'] = np.nan
    c7_expanded.loc[valid_idx, 'q_global'] = qvals

c7_expanded['q_stratified'] = np.nan
for (tissue, lin), group in c7_expanded.groupby(['tissue', 'lineage']):
    vm = group['p_value'].notna()
    vi = group.index[vm]
    if len(vi) < 2: continue
    _, qv, _, _ = multipletests(c7_expanded.loc[vi, 'p_value'].values, alpha=0.05, method='fdr_bh')
    c7_expanded.loc[vi, 'q_stratified'] = qv

# Save (overwrite C7)
c7_expanded.to_csv(os.path.join(RESULTS_V2, 'C7/C7_correlations.csv'), index=False)

# Also update C9B
c7_expanded.to_csv(os.path.join(RESULTS_V2, 'C9B/C7_FDR.csv'), index=False)

global_surv = (c7_expanded['q_global'] < 0.05).sum()
strat_surv = (c7_expanded['q_stratified'] < 0.05).sum()
sig_nominal = (c7_expanded['p_value'] < 0.05).sum()

print(f"\nC7 EXPANDED complete ({time.time()-t0:.0f}s)")
print(f"  Genes: {len(C7_GENES_FOUND)}")
print(f"  Pairs tested: {pairs_tested}")
print(f"  Nominal sig: {sig_nominal}")
print(f"  Global FDR survived: {global_surv}")
print(f"  Stratified FDR survived: {strat_surv}")

print(f"\nTop 10 correlations:")
top10 = c7_expanded.sort_values('rho', ascending=False).head(10)
for _, r in top10.iterrows():
    q_str = f"q={r['q_global']:.4f}" if not np.isnan(r.get('q_global', np.nan)) else ""
    print(f"  {r['gene1']:<12} ↔ {r['gene2']:<12} {r['tissue']}/{r['lineage']:<10} "
          f"ρ={r['rho']:.3f} p={r['p_value']:.4f} {q_str}")

# Key correlations from manuscript
print(f"\nKey manuscript correlations check:")
key_pairs = [
    ('TFAM', 'DNMT1', 'Blood', 'Myeloid'),
    ('JAK1', 'TGFBR2', 'Blood', 'CD4_T'),
    ('MT-CYB', 'MT-ND1', 'Blood', 'CD8_T'),
    ('TGFB1', 'LGALS9', 'Blood', 'Myeloid'),
    ('SOCS1', 'JAK1', 'Blood', 'CD4_T'),
]
for g1, g2, tissue, lin in key_pairs:
    row = c7_expanded[((c7_expanded['gene1']==g1)&(c7_expanded['gene2']==g2)&
                        (c7_expanded['tissue']==tissue)&(c7_expanded['lineage']==lin)) |
                       ((c7_expanded['gene1']==g2)&(c7_expanded['gene2']==g1)&
                        (c7_expanded['tissue']==tissue)&(c7_expanded['lineage']==lin))]
    if len(row) > 0:
        r = row.iloc[0]
        print(f"  ✅ {g1}↔{g2} {tissue}/{lin}: ρ={r['rho']:.3f} p={r['p_value']:.4f}")
    else:
        print(f"  ❌ {g1}↔{g2} {tissue}/{lin}: NOT FOUND")


C7 EXPANDED complete (487s)
  Genes: 58
  Pairs tested: 19836
  Nominal sig: 4590
  Global FDR survived: 2020
  Stratified FDR survived: 2104

Top 10 correlations:
  CTLA4        ↔ LAYN         Liver/B          ρ=0.990 p=0.0000 q=0.0000
  HLA-DPA1     ↔ HLA-DRA      Liver/PlasmaB    ρ=0.980 p=0.0000 q=0.0000
  HLA-DPA1     ↔ HLA-DPB1     Liver/PlasmaB    ρ=0.973 p=0.0000 q=0.0000
  CD74         ↔ HLA-DRA      Liver/PlasmaB    ρ=0.972 p=0.0000 q=0.0000
  HLA-DPA1     ↔ HLA-DPB1     Blood/Myeloid    ρ=0.970 p=0.0000 q=0.0000
  HLA-DRA      ↔ HLA-DRB1     Liver/PlasmaB    ρ=0.970 p=0.0000 q=0.0000
  CD74         ↔ HLA-DPB1     Blood/Myeloid    ρ=0.970 p=0.0000 q=0.0000
  FCER1G       ↔ GZMB         Blood/PlasmaB    ρ=0.968 p=0.0000 q=0.0000
  CD74         ↔ HLA-DRB1     Liver/PlasmaB    ρ=0.968 p=0.0000 q=0.0000
  FCER1G       ↔ TYROBP       Blood/PlasmaB    ρ=0.967 p=0.0000 q=0.0000

Key manuscript correlations check:
  ✅ TFAM↔DNMT1 Blood/Myeloid: ρ=0.925 p=0.0000
  ✅ JAK1↔TGFBR2 Blood/

In [ ]:
### 'Six Concurrent Layers of Effector Suppression in the Immune Tolerant Phase of Chronic Hepatitis B' 이 타이틀은 살아남을 수 있는건가? 검

In [59]:
# ========================================
# 6-LAYER SURVIVAL CHECK: v2 data
# ========================================
c3_blood = pd.read_csv(os.path.join(RESULTS_V2, 'C3/C3_genes_blood.csv'))
c3_liver = pd.read_csv(os.path.join(RESULTS_V2, 'C3/C3_genes_liver.csv'))
c5_blood = pd.read_csv(os.path.join(RESULTS_V2, 'C5/C5_genes_blood.csv'))
c5_liver = pd.read_csv(os.path.join(RESULTS_V2, 'C5/C5_genes_liver.csv'))

# Combine C3+C5 for comprehensive check
all_blood = pd.concat([c3_blood, c5_blood]).drop_duplicates(subset=['gene','lineage','comparison'])
all_liver = pd.concat([c3_liver, c5_liver]).drop_duplicates(subset=['gene','lineage','comparison'])

print("="*80)
print("SIX-LAYER SURVIVAL CHECK (v2 donor-corrected)")
print("="*80)

six_layer_genes = [
    # Layer 1: Myeloid paracrine suppression
    ('L1', 'TGFB1',   'Myeloid',  'Blood'),
    ('L1', 'LGALS9',  'Myeloid',  'Blood'),
    ('L1', 'LILRB1',  'Myeloid',  'Blood'),
    ('L1', 'SIGLEC10','Myeloid',  'Blood'),
    ('L1', 'AIM2',    'Myeloid',  'Blood'),
    ('L1', 'MEFV',    'Myeloid',  'Blood'),
    ('L1', 'PYCARD',  'Myeloid',  'Blood'),

    # Layer 2: Epigenetic silencing
    ('L2', 'DNMT1',   'Myeloid',  'Blood'),
    ('L2', 'DNMT3A',  'Myeloid',  'Blood'),
    ('L2', 'TET2',    'Myeloid',  'Blood'),
    ('L2', 'DNMT1',   'Myeloid',  'Liver'),

    # Layer 3: Metabolic checkpoint
    ('L3', 'MTOR',    'Myeloid',  'Blood'),
    ('L3', 'MTOR',    'Myeloid',  'Liver'),
    ('L3', 'LDHA',    'Myeloid',  'Blood'),
    ('L3', 'LDHA',    'Myeloid',  'Liver'),
    ('L3', 'TFAM',    'Myeloid',  'Blood'),
    ('L3', 'TFAM',    'CD4_T',    'Blood'),

    # Layer 4: JAK-STAT paradox
    ('L4', 'JAK1',    'Myeloid',  'Blood'),
    ('L4', 'STAT1',   'Myeloid',  'Blood'),
    ('L4', 'SOCS1',   'CD4_T',    'Blood'),
    ('L4', 'SOCS1',   'CD8_T',    'Blood'),
    ('L4', 'SOCS3',   'CD4_T',    'Blood'),

    # Layer 5: Liver T cell exhaustion
    ('L5', 'TOX',     'CD4_T',    'Liver'),
    ('L5', 'TOX',     'CD8_T',    'Liver'),
    ('L5', 'TOX2',    'CD4_T',    'Liver'),
    ('L5', 'LAYN',    'CD4_T',    'Liver'),
    ('L5', 'CTLA4',   'CD4_T',    'Liver'),
    ('L5', 'TIGIT',   'CD4_T',    'Liver'),
    ('L5', 'TIGIT',   'CD8_T',    'Liver'),

    # Layer 6: Terminal differentiation block
    ('L6', 'PRDM1',   'CD4_T',    'Liver'),
    ('L6', 'PRDM1',   'CD4_T',    'Blood'),
    ('L6', 'PRDM1',   'CD8_T',    'Blood'),
    ('L6', 'RORC',    'CD4_T',    'Liver'),
]

print(f"\n{'Layer':<5} {'Gene':<10} {'Lineage':<10} {'Tissue':<7} {'pct':>8} {'p':>8} {'sig':>4} {'consist':>10} | v2 Status")
print("-"*85)

layer_status = {}
for layer, gene, lin, tissue in six_layer_genes:
    df = all_blood if tissue == 'Blood' else all_liver
    row = df[(df['gene']==gene) & (df['lineage']==lin) & (df['comparison']=='NL→IT')]

    if len(row) == 0:
        print(f"{layer:<5} {gene:<10} {lin:<10} {tissue:<7} {'N/A':>8} {'N/A':>8} {'':>4} {'':>10} | ❌ NOT FOUND")
        continue

    r = row.iloc[0]
    p = r['p_value']
    pct = r['pct_change']
    sig = '★' if p < 0.05 else ('†' if p < 0.10 else 'NS')
    consist = r.get('consistency', '')

    status = '✅ SIG' if p < 0.05 else ('⚠️ TREND' if p < 0.10 else '❌ NS')
    print(f"{layer:<5} {gene:<10} {lin:<10} {tissue:<7} {pct:>+7.1f}% {p:>8.4f} {sig:>4} {consist:>10} | {status}")

    if layer not in layer_status:
        layer_status[layer] = {'sig': 0, 'trend': 0, 'ns': 0, 'total': 0}
    layer_status[layer]['total'] += 1
    if p < 0.05: layer_status[layer]['sig'] += 1
    elif p < 0.10: layer_status[layer]['trend'] += 1
    else: layer_status[layer]['ns'] += 1

print(f"\n{'='*60}")
print("LAYER SURVIVAL SUMMARY")
print(f"{'='*60}")
for layer in ['L1','L2','L3','L4','L5','L6']:
    s = layer_status.get(layer, {'sig':0,'trend':0,'ns':0,'total':0})
    pct_sig = s['sig']/s['total']*100 if s['total']>0 else 0
    verdict = '✅ STRONG' if pct_sig >= 50 else ('⚠️ WEAK' if pct_sig >= 25 else '❌ FAILED')
    print(f"  {layer}: {s['sig']}/{s['total']} sig ({pct_sig:.0f}%), {s['trend']} trends, {s['ns']} NS → {verdict}")

SIX-LAYER SURVIVAL CHECK (v2 donor-corrected)

Layer Gene       Lineage    Tissue       pct        p  sig    consist | v2 Status
-------------------------------------------------------------------------------------
L1    TGFB1      Myeloid    Blood    +155.6%   0.0079    ★      25/25 | ✅ SIG
L1    LGALS9     Myeloid    Blood     +93.7%   0.0159    ★      24/25 | ✅ SIG
L1    LILRB1     Myeloid    Blood    +101.8%   0.0317    ★      23/25 | ✅ SIG
L1    SIGLEC10   Myeloid    Blood    +137.7%   0.0317    ★      23/25 | ✅ SIG
L1    AIM2       Myeloid    Blood   +1431.2%   0.0112    ★      25/25 | ✅ SIG
L1    MEFV       Myeloid    Blood    +201.1%   0.0079    ★      25/25 | ✅ SIG
L1    PYCARD     Myeloid    Blood     +24.7%   0.0079    ★      25/25 | ✅ SIG
L2    DNMT1      Myeloid    Blood    +163.8%   0.0079    ★      25/25 | ✅ SIG
L2    DNMT3A     Myeloid    Blood    +348.3%   0.0119    ★      25/25 | ✅ SIG
L2    TET2       Myeloid    Blood    +150.1%   0.0079    ★      25/25 | ✅ SIG
L2   